# Dunnhumby M3: 이진 구매 전파 + N/V 변화분 (개발 시드 43)

기존 M3 두 축 arm B의 **동일한 구매쌍·N/V 엣지가중치**를 사용합니다. M1의 이진 정규화 전파행렬을 `P0`, 기존 M3를 `PNV`라 하면 이번 조건은 `P = P0 + 0.5(PNV-P0)`입니다. 두 경로를 따로 학습·동결하거나 점수를 나중에 더하지 않습니다. 혼합행렬 한 개로 처음부터 공동학습합니다.

앞서 제안한 방식과 ‘M1 이진 전파를 남기고 N/V 변화분만 더하는 방식’은 **같은 수식**이므로 GPU에서 두 번 학습하지 않습니다. 기존 `PNV` 결과는 재사용하고, 새 혼합 조건 **한 arm만** 학습합니다.

고정: Dunnhumby 역사적 개발 684–690일, seed 43, 300 epoch, 25마다 validation 기록. 최종 비교는 사전 고정한 300 epoch, 100 epoch은 진단입니다. M1과 기존 N/V M3는 완료 결과를 재사용합니다. 신규상품 추천·학습쌍 제외·카탈로그 보존·uniform 음성 1개·plain BPR이며 test/holdout은 열지 않습니다. 단일 반복 노출 개발시드이므로 성공·유의성·일반화는 주장하지 않습니다.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
from google.colab import drive

ROOT = Path('/content/drive/MyDrive/논문/data')
M1_DIR = ROOT/'results_v3_dunnhumby_clv_m2_capacity_search_v1'
OLD_M3_DIR = ROOT/'results_v3_dunnhumby_clv_m3_centered_value_graph_v1'
if not M1_DIR.is_dir() or not list(OLD_M3_DIR.glob('clv_m3_centered_value_graph_*.json')):
    if not os.path.ismount('/content/drive'):
        try:
            drive.mount('/content/drive')
        except (ValueError, NotImplementedError) as exc:
            raise RuntimeError('Drive 연결 실패. 학습을 시작하지 않았습니다.') from exc
    if not M1_DIR.is_dir() or not list(OLD_M3_DIR.glob('clv_m3_centered_value_graph_*.json')):
        raise RuntimeError('M1 또는 기존 N/V M3 완료 결과가 없습니다. 재학습하지 않고 중단합니다.')

SOURCE_COMMIT = 'b7227ba4bbe3d117061650421ee2511fc4c7384a'
REPO = Path('/content/clv-m3-binary-residual-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
for name in ('lightgcn_clv_v3', 'lightgcn_clv_m3_centered_value_graph', 'clv_m3_binary_residual_nv_screen'):
    if name in sys.modules and Path(sys.modules[name].__file__).resolve().parent != REPO.resolve():
        raise RuntimeError(f'{name}이 다른 코드에서 로드됐습니다. 런타임을 다시 시작하세요.')
os.chdir(REPO)
sys.path.insert(0, str(REPO))
import torch
assert torch.cuda.is_available(), '학습에는 GPU 런타임을 사용하세요.'
import clv_m3_binary_residual_nv_screen as screen
screen.self_test()
cfg = screen.configure(m1_result_dir=str(M1_DIR))
print(json.dumps({'source_commit': SOURCE_COMMIT, 'seed': cfg.seeds, 'epochs': cfg.epochs,
                  'alpha': screen.ALPHA, 'out_dir': cfg.out_dir}, ensure_ascii=False, indent=2))

## 실행

실행 전에 M1 결과와 기존 N/V M3의 학습 설정·강도·시드가 맞는지 확인합니다. 새 arm은 epoch마다 optimizer·난수상태를 Drive에 저장하므로 같은 코드·설정으로 재실행하면 이어집니다. GPU 학습은 새 혼합 M3 **한 번**뿐입니다.

In [ ]:
result = screen.run(cfg)

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
key = ['recall@10', 'ndcg@10', 'recall@20', 'ndcg@20', 'recall@50', 'ndcg@50',
       'price_purchase_amount_weighted_hit@10', 'vndcg@10',
       'price_purchase_amount_weighted_hit@20', 'vndcg@20',
       'price_purchase_amount_weighted_hit@50', 'vndcg@50']
view = result['comparison']
print(view[view.metric.isin(key)].to_string(index=False))
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))
zip_path = Path('/content/clv_m3_binary_residual_nv_seed43_results.zip')
with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as archive:
    for path in result['paths'].values():
        archive.write(path, arcname=Path(path).name)
files.download(str(zip_path))